# Exercise 2: M5 Data Preparation

In this notebook, I'm getting the raw M5 data ready for analysis. The CSV files are quite large, so I'm converting them to **Parquet** format. This makes them much faster to load in the next assignments. I'm also creating a small summary of the data schemas so I can refer back to it later.

In [1]:
import json
import time

import pandas as pd

from tis3il.m5_utils import PROCESSED, ensure_dirs, find_raw_csv

FILES = {
    "calendar.csv": "calendar.parquet",
    "sell_prices.csv": "sell_prices.parquet",
    "sales_train_validation.csv": "sales_train_validation.parquet",
    "sales_train_evaluation.csv": "sales_test_evaluation.parquet",
}

def describe_frame(name: str, df: pd.DataFrame) -> dict[str, object]:
    return {
        "file": name,
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
        "column_names": list(df.columns),
        "dtypes": {column: str(dtype) for column, dtype in df.dtypes.items()},
    }

ensure_dirs()
summary = []
for csv_name, parquet_name in FILES.items():
    started = time.perf_counter()
    df = pd.read_csv(find_raw_csv(csv_name))
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
    target = PROCESSED / parquet_name
    df.to_parquet(target, index=False)
    item = describe_frame(csv_name, df)
    item["write_seconds"] = round(time.perf_counter() - started, 3)
    item["parquet"] = str(target)
    summary.append(item)

(PROCESSED / "exercise2_data_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
pd.DataFrame(summary)[["file", "rows", "columns", "write_seconds", "parquet"]]

,file,rows,columns,write_seconds,parquet
0,calendar.csv,1969,14,0.371,C:\Users\EliasTeubner\Documents\Elias-FH\TIS3I...
1,sell_prices.csv,6841121,4,3.414,C:\Users\EliasTeubner\Documents\Elias-FH\TIS3I...
2,sales_train_validation.csv,30490,1919,4.098,C:\Users\EliasTeubner\Documents\Elias-FH\TIS3I...
3,sales_train_evaluation.csv,30490,1947,4.280,C:\Users\EliasTeubner\Documents\Elias-FH\TIS3I...
